In [ ]:
!pip install pandas scikit-learn xgboost matplotlib joblib
!python lung_baseline.py --data "survey_lung_cancer.csv"

Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive

               model  cv_auc_mean  cv_auc_std  test_auc  sensitivity  specificity  brier_score
Logistic Regression        0.566       0.033     0.648        0.458        0.750        0.250
      Random Forest        0.552       0.081     0.648        0.688        0.625        0.211
            XGBoost        0.566       0.091     0.637        0.583        0.750        0.239

Best model by CV AUC: Logistic Regression (saved to models/lung_model.joblib)


In [ ]:
from google.colab import files
import os

if not os.path.exists('survey lung cancer.csv'):
    print("Please upload 'survey lung cancer.csv' using the upload button below:")
    uploaded = files.upload()
else:
    print("Dataset 'survey lung cancer.csv' already exists in the workspace!")

Please upload 'survey lung cancer.csv' using the upload button below:


Saving archive (1).zip to archive (1).zip


In [ ]:
!unzip -o "archive (1).zip"
!python lung_baseline.py --data "survey_lung_cancer.csv"

Archive:  archive (1).zip
  inflating: survey_lung_cancer.csv  
  inflating: lung_baseline.py        
  inflating: lung_explain.py         
Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive

               model  cv_auc_mean  cv_auc_std  test_auc  sensitivity  specificity  brier_score
Logistic Regression        0.566       0.033     0.648        0.458        0.750        0.250
      Random Forest        0.552       0.081     0.648        0.688        0.625        0.211
            XGBoost        0.566       0.091     0.637        0.583        0.750        0.239

Best model by CV AUC: Logistic Regression (saved to models/lung_model.joblib)


In [ ]:
!pip install shap
!mkdir -p models
!cp lung_cancer_model.joblib models/lung_model.joblib
!python lung_explain.py --data "survey_lung_cancer.csv" --row 0
!python lung_explain.py --data "survey_lung_cancer.csv" --summary-plot

cp: cannot stat 'lung_cancer_model.joblib': No such file or directory
Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive
Background dataset has 276 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=276 when initializing the masker.
{
  "traffic_light": "yellow",
  "people_out_of_100": 53,
  "headline": "Your score is somewhat higher than average.",
  "uncertain": true,
  "uncertainty_note": "The tool is not confident about this result. Please do not rely on it; see a doctor.",
  "top_reasons": [
    "Your age (69) was linked to a higher score; lung cancer becomes more common with age."
  ],
  "working_in_your_favour": [],
  "next_step": "Consider a check-up and ask your doctor whether lung screening makes sense for you.",
  "emergency_note": "If you cough up blood, have severe trouble breathing, or sudden severe chest pain, call your local emergency number now.",
  "scale_no

In [ ]:
%%writefile app.py
import os
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

from lung_baseline import TARGET, load_clean
from lung_explain import (LABELS, build_story, contributions_occlusion,
                          predict_p)

CSV_PATH = os.environ.get("LUNG_CSV", "survey_lung_cancer.csv")
MODEL_PATH = "models/lung_model.joblib"

COLORS = {"green": "#2e9e5b", "yellow": "#e0a800", "red": "#d64545"}
EMOJI = {"green": "🟢", "yellow": "🟡", "red": "🔴"}

# One question per screen: (feature, question, why we ask)
QUESTIONS = [
    ("AGE", "How old are you?",
     "Lung cancer becomes more common as people get older."),
    ("GENDER", "What is your sex?",
     "The survey this tool learned from recorded it. It is not shown as a reason in your result."),
    ("SMOKING", "Do you smoke?",
     "Smoking is the biggest known cause of lung cancer."),
    ("YELLOW_FINGERS", "Do you have yellow-stained fingers?",
     "This can be a sign of heavy smoking."),
    ("COUGHING", "Do you have a cough that has lasted more than 3 weeks?",
     "A cough that does not go away is one of the most common early signs doctors want to check."),
    ("WHEEZING", "Do you wheeze (a whistling sound when you breathe)?",
     "Wheezing can be a sign of a problem in the airways."),
    ("SHORTNESS_OF_BREATH", "Do you get short of breath doing normal things?",
     "Doctors want to look into unexplained breathlessness."),
    ("CHEST_PAIN", "Do you have chest pain?",
     "Doctors want to look into chest pain that keeps coming back."),
    ("SWALLOWING_DIFFICULTY", "Do you have trouble swallowing?",
     "Doctors want to look into this symptom."),
    ("FATIGUE", "Are you tired much of the time, without a clear reason?",
     "It has many causes, so on its own it is not a warning sign."),
    ("CHRONIC_DISEASE", "Do you have a long-term health condition?",
     "The survey data linked this to a higher score."),
    ("ALLERGY", "Do you have allergies?",
     "The survey data linked this to a higher score; the connection is not well understood."),
    ("ALCOHOL_CONSUMING", "Do you drink alcohol regularly?",
     "The survey data linked this to a higher score."),
    ("ANXIETY", "Do you often feel anxious?",
     "The survey data linked this to a higher score. It is not a known cause of lung cancer."),
    ("PEER_PRESSURE", "Do you often feel peer pressure?",
     "The survey data linked this to a higher score. It is not a known cause of lung cancer."),
]

# What-if toggles shown on the result page: (feature, text)
WHAT_IFS = [
    ("SMOKING", "What if you did not smoke?"),
    ("COUGHING", "What if you had no long-lasting cough?"),
    ("WHEEZING", "What if you did not wheeze?"),
    ("SHORTNESS_OF_BREATH", "What if you were not short of breath?"),
]


# ---- Model + scoring --------------------------------------------------------
@st.cache_resource
def load_everything():
    bundle = joblib.load(MODEL_PATH)
    df = load_clean(CSV_PATH)
    features = bundle["features"]
    background = df.drop(columns=[TARGET])[features]
    return bundle["model"], features, background


def score(model, features, answers):
    row = pd.DataFrame([[answers[f] for f in features]], columns=features)
    return float(predict_p(model, row)[0]), row


def make_story(model, features, background, answers):
    p, row = score(model, features, answers)
    contribs = contributions_occlusion(model, background, row)
    return p, build_story(p, contribs, features, answers)


# ---- Drawing helpers --------------------------------------------------------
def people_grid(n_colored, color):
    """100 little people; the first n_colored are filled in."""
    dots = []
    for i in range(100):
        c = color if i < n_colored else "#d0d4da"
        dots.append(
            f'<span style="display:inline-block;width:8.6%;text-align:center;'
            f'font-size:1.5rem;color:{c};line-height:1.25">●</span>'
        )
    return f'<div style="max-width:420px">{"".join(dots)}</div>'


def init_state():
    if "step" not in st.session_state:
        st.session_state.step = -1  # -1 welcome, 0..N-1 questions, N result
        st.session_state.answers = {}


def ask(feature, question, why):
    st.progress((st.session_state.step + 1) / len(QUESTIONS))
    st.subheader(question)
    if feature == "AGE":
        value = st.number_input("Age", 18, 100, 50, label_visibility="collapsed")
    elif feature == "GENDER":
        choice = st.radio("Sex", ["Male", "Female"], horizontal=True, label_visibility="collapsed")
        value = 1 if choice == "Male" else 0
    else:
        choice = st.radio("Answer", ["No", "Yes"], horizontal=True, label_visibility="collapsed")
        value = 1 if choice == "Yes" else 0
    with st.expander("Why are we asking this?"):
        st.write(why)
    return int(value)


# ---- Screens ------------------------------------------------------------------
def screen_welcome():
    st.title("Lung health check")
    st.write("Answer a few simple questions and get a plain-language picture of how your "
             "answers compare with people in a health survey.")
    st.info("This is a learning project, **not** a medical test and not a diagnosis. "
            "It takes about 2 minutes. Nothing you enter is saved.")
    if st.button("Start", type="primary"):
        st.session_state.step = 0
        st.rerun()


def screen_question():
    i = st.session_state.step
    feature, question, why = QUESTIONS[i]
    value = ask(feature, question, why)
    left, right = st.columns(2)
    if left.button("Back", disabled=(i == 0)):
        st.session_state.step -= 1
        st.rerun()
    if right.button("Next" if i < len(QUESTIONS) - 1 else "See my result", type="primary"):
        st.session_state.answers[feature] = value
        st.session_state.step += 1
        st.rerun()


def screen_result(model, features, background):
    answers = st.session_state.answers
    p, story = make_story(model, features, background, answers)
    band = story["traffic_light"]

    st.title(f"{EMOJI[band]} {story['headline']}")
    if story["uncertain"]:
        st.warning(story["uncertainty_note"])

    st.markdown(f"**Out of 100 people with answers like yours in this survey, "
                f"about {story['people_out_of_100']} were in the lung cancer group.**")
    st.markdown(people_grid(story["people_out_of_100"], COLORS[band]), unsafe_allow_html=True)
    st.caption(story["scale_note"])

    if story["top_reasons"]:
        st.subheader("What pushed your score up")
        for r in story["top_reasons"]:
            st.write(f"• {r}")
    if story["working_in_your_favour"]:
        st.subheader("What's working in your favour")
        for r in story["working_in_your_favour"]:
            st.write(f"• {r}")

    # What-if: only offer changes that apply to this person
    options = [(f, t) for f, t in WHAT_IFS if answers.get(f) == 1]
    if options:
        st.subheader("What if…?")
        chosen = [f for f, t in options if st.toggle(t, key=f"whatif_{f}")]
        if chosen:
            changed = dict(answers)
            for f in chosen:
                changed[f] = 0
            p2, _ = score(model, features, changed)
            st.markdown(f"With those changes, about **{int(round(p2 * 100))} out of 100**.")
            st.markdown(people_grid(int(round(p2 * 100)), COLORS[risk_color(p2)]),
                        unsafe_allow_html=True)
            st.caption("This shows how the tool's score changes. It does not promise "
                       "that making a change would change your real health.")

    st.subheader("Next step")
    st.success(story["next_step"])
    st.error(story["emergency_note"])
    st.caption(story["disclaimer"])

    if st.button("Start again"):
        st.session_state.step = -1
        st.session_state.answers = {}
        st.rerun()


def risk_color(p):
    from lung_explain import risk_band
    return risk_band(p)


def main():
    st.set_page_config(page_title="Lung health check", page_icon="🫁", layout="centered")
    init_state()
    if not Path(MODEL_PATH).exists():
        st.error(f"Could not find {MODEL_PATH}. Run lung_baseline.py first.")
        return
    model, features, background = load_everything()

    step = st.session_state.step
    if step == -1:
        screen_welcome()
    elif step < len(QUESTIONS):
        screen_question()
    else:
        screen_result(model, features, background)


if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
# 1. Install localtunnel to expose the port
!npm install -g localtunnel

# 2. Get your public IP address (needed as a password for localtunnel security)
!curl ipv4.icanhazip.com

# 3. Start Streamlit in the background and expose port 8501
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼35.225.204.49
/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://tangy-weeks-tickle.loca.lt
^C


In [ ]:
import os

# 1. Install streamlit since it is missing in this environment
!pip install streamlit

# 2. Extract files from the uploaded archive (1).zip instead of archive (2).zip
!unzip -o "archive (1).zip"

# 3. Create models directory and run baseline to train the model
!mkdir -p models
!python lung_baseline.py --data "survey_lung_cancer.csv"

# 4. Run explain scripts to test they work before launching
!python lung_explain.py --data "survey_lung_cancer.csv" --row 0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.8 MB/s eta 0:00:00
Archive:  archive (1).zip
  inflating: survey_lung_cancer.csv  
  inflating: lung_baseline.py        
  inflating: lung_explain.py         
Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive

               model  cv_auc_mean  cv_auc_std  test_auc  sensitivity  specificity  brier_score
Logistic Regression        0.566       0.033     0.648        0.458        0.750        0.250
      Random Forest        0.552       0.081     0.648        0.688        0.625        0.211
            XGBoost        0.566       0.091     0.637        0.583        0.750        0.239

Best model by CV AUC: Logistic Regression (saved to models/lung_model.joblib)
Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive
Background dataset has 276 samples but max_samples=100. S

In [ ]:
# 5. Start Streamlit in the background and expose port 8501
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧

2026-10-21 17:59:39.189 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.225.204.49:8501

your url is: https://blue-pans-heal.loca.lt
Rows: 309 -> 276 after removing 33 duplicates
Class balance (1 = lung cancer): 86.2% positive
  Stopping...
^C
